# A2 — Extended-sample build (2022–2026)

Builds the *extended* dataset alongside the base sample, without touching any
base file. The thesis's primary results are written on the base sample
(2011–2022); this extension (Aug 2022 onward) exists to support the
post-COVID era comparison in **B3**.

**Design: extend alongside, never replace.** Every extended table is written
to a `*_ext` location, physically separate from its base counterpart. The
analysis layer reads either sample via a single switch —
`event_window_profile.set_sample("base" | "ext")` — so no analysis code is
duplicated.

**Prerequisites.** The extended raw data must already be in place:
- CBOE daily zips for 2022-08-01 onward, in `CBOE_Data_2011_2026/`
- A fresh CRSP daily export (2022–2025) as a CSV in `data/crsp/`
- IBES is unchanged — the existing `ibes_clean.parquet` already covers
  2011–2026, so no extended IBES build is needed; the extended event panel
  is built from the same IBES file, date-filtered by the extended CBOE window.

**Run A1 first** (it builds the base sample). This notebook assumes the base
`*.parquet` files exist, since several steps verify the extension against
them.

Usable extended window: **2022-08-01 to Feb 2026** for events (capped by
IBES actuals), and moneyness/size controls run to **2025-12-31** (capped by
CRSP). Options data itself runs to May 2026 but has no spot price or earnings
event beyond those ceilings.

## Setup

In [1]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        if (current / "src").exists():
            return current / "src"
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

sys.path.append(str(find_src_dir()))

import polars as pl
from pathlib import Path
from paths import DATA_DIR, IBES_DIR, CRSP_DIR

## Migration record (one-time, documentary)

The steps below were performed once to stage the extended raw data. They are
recorded here for transparency; a clean rebuild from the full raw archive
does not require repeating them exactly, but the isolation logic — extracting
/ ingesting only the *new* files to a separate location — is what keeps the
extended outputs uncontaminated by base data.

**CBOE zips.** The full zip archive (`CBOE_Data_2011_2026/`, 3,823 files)
contains both base (2011–2022) and new (2022-08-01–2026) periods. Only the
961 new zips were extracted, to a separate folder, then ingested to
`cboe_parquet_ext/`. There is a genuine ~11-week gap in source coverage
between the base end (2022-05-16) and the extension start (2022-08-01) — this
is a gap in the CBOE export, not a processing artefact.

**CRSP.** The new CRSP CSV was isolated into `data/crsp/ext_source/` so the
ingestion glob would read only it, not the base CRSP CSV also present in the
folder:

```python
import shutil
crsp_dir = Path(DATA_DIR / "crsp")
ext_dir = crsp_dir / "ext_source"
ext_dir.mkdir(exist_ok=True)
shutil.move(str(crsp_dir / "<new_crsp_file>.csv"), str(ext_dir / "<new_crsp_file>.csv"))
```

Every ingestion/build call below writes to a `*_ext` output and leaves the
base file untouched.

## Step 1 — Extended CBOE options → Parquet

Extract the new zips (2022-08-01 onward) and ingest to `cboe_parquet_ext/`.
The extraction is pointed at a folder containing only the new zips so it
cannot re-extract the base period. If the extended parquet already exists,
this skips.

In [2]:
from pipeline.extract_zips import run_extraction
from pipeline.ingest_cboe import run_ingestion

ext_parquet = DATA_DIR / "cboe_parquet_ext"
if not any(ext_parquet.glob("*.parquet")):
    # extraction: new zips were isolated to CBOE_Data_2022_2026_new/ first
    run_extraction(
        zip_dir=Path(r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\CBOE_Data_2022_2026_new"),
        dest_dir=DATA_DIR / "CBOE_DATA_RAW_EXTRACTED_EXT",
    )
    run_ingestion(
        raw_dir=str(DATA_DIR / "CBOE_DATA_RAW_EXTRACTED_EXT"),
        out_dir=str(ext_parquet),
    )
else:
    print("cboe_parquet_ext already exists -- skipping.")

cboe_parquet_ext already exists -- skipping.


### Verify: extended CBOE schema matches base

A differently-named column here would silently break the `cust_`/`procust_`
aggregation downstream. Both sets must be empty.

In [3]:
base = pl.scan_parquet(DATA_DIR / "cboe_parquet" / "cboe_openclose_2021.parquet").collect_schema().names()
ext  = pl.scan_parquet(DATA_DIR / "cboe_parquet_ext" / "cboe_openclose_2024.parquet").collect_schema().names()
print("only in base:", set(base) - set(ext))
print("only in ext: ", set(ext) - set(base))

only in base: set()
only in ext:  set()


## Step 2 — Extended dispersion events

Built from the shared IBES file against the *extended* CBOE date window. The
`cboe_dir` argument is what makes the event window derive from 2022–2026
rather than the base period — watch the printed `CBOE data window:` line, it
must read 2022-08-01 onward. Writes `dispersion_events_ext.parquet`; the base
`dispersion_events.parquet` is untouched.

In [4]:
from pipeline.build_dispersion_events import build_dispersion_events

ext_events = IBES_DIR / "dispersion_events_ext.parquet"
if not ext_events.exists():
    build_dispersion_events(
        cboe_dir=DATA_DIR / "cboe_parquet_ext",
        out_path=ext_events,
    )
else:
    print("dispersion_events_ext.parquet already exists -- skipping.")

dispersion_events_ext.parquet already exists -- skipping.


### Verify: both event panels coexist, extension is dated correctly

Base must still show ~163k events spanning 2011–2022; ext a smaller count
spanning 2022–2026 with earnings volatility populated (the rolling measure
reaches back through the shared IBES history, so early-window events still
get it).

In [5]:
for f in ["dispersion_events.parquet", "dispersion_events_ext.parquet"]:
    ev = pl.read_parquet(IBES_DIR / f)
    print(f"{f}: {ev.height:,} events, "
          f"ANNDATS_ACT {ev['ANNDATS_ACT'].min()} to {ev['ANNDATS_ACT'].max()}, "
          f"earnings_vol non-null: {ev['earnings_volatility'].drop_nulls().len():,}")

dispersion_events.parquet: 163,019 events, ANNDATS_ACT 2011-01-03 to 2022-05-16, earnings_vol non-null: 137,886
dispersion_events_ext.parquet: 53,557 events, ANNDATS_ACT 2022-08-01 to 2026-02-19, earnings_vol non-null: 35,992


## Step 3 — Extended daily participant activity

From `cboe_parquet_ext/` to a separate `cboe_daily_retail_ext/` folder. Same
26-column output as base (16 tier/vol columns + 5 qty totals + firm/bd/mm
totals).

In [6]:
from pipeline.build_daily_retail_activity import build_daily_retail_activity

ext_daily = DATA_DIR / "cboe_daily_retail_ext"
if not list(ext_daily.glob("*.parquet")):
    build_daily_retail_activity(
        cboe_dir=DATA_DIR / "cboe_parquet_ext",
        out_dir=ext_daily,
    )
else:
    print("cboe_daily_retail_ext already exists -- skipping.")

cboe_daily_retail_ext already exists -- skipping.


### Verify: column parity with base, qty columns present

In [7]:
base = pl.read_parquet(DATA_DIR / "cboe_daily_retail" / "daily_retail_2021.parquet")
ext  = pl.read_parquet(DATA_DIR / "cboe_daily_retail_ext" / "daily_retail_2023.parquet")
print("column diff:", set(base.columns) ^ set(ext.columns))
print("ext qty cols:", [c for c in ext.columns if c.endswith("_qty_total")])
print("base files:", len(list((DATA_DIR/"cboe_daily_retail").glob("*.parquet"))),
      "| ext files:", len(list((DATA_DIR/"cboe_daily_retail_ext").glob("*.parquet"))))

column diff: set()
ext qty cols: ['firm_qty_total', 'bd_qty_total', 'mm_qty_total', 'retail_qty_total', 'procust_qty_total']
base files: 12 | ext files: 5


## Step 4 — Extended CRSP prices

The new CRSP CSV, isolated to `crsp/ext_source/`, ingested to
`crsp_daily_ext.parquet`. Covers 2022–2025 (the CRSP ceiling). Base
`crsp_daily.parquet` untouched.

In [8]:
from pipeline.ingest_crsp import run_crsp_ingestion

ext_crsp = CRSP_DIR / "crsp_daily_ext.parquet"
if not ext_crsp.exists():
    run_crsp_ingestion(
        crsp_dir=CRSP_DIR / "ext_source",
        out_path=ext_crsp,
    )
else:
    print("crsp_daily_ext.parquet already exists -- skipping.")

crsp_daily_ext.parquet already exists -- skipping.


### Verify: both CRSP files coexist, extension dated 2022–2025

In [9]:
for f in ["crsp_daily.parquet", "crsp_daily_ext.parquet"]:
    lf = pl.scan_parquet(CRSP_DIR / f).select(
        pl.col("DlyCalDt").min().alias("min"),
        pl.col("DlyCalDt").max().alias("max"),
        pl.len().alias("rows"),
    ).collect()
    print(f"{f}: {lf['rows'][0]:,} rows, {lf['min'][0]} to {lf['max'][0]}")

crsp_daily.parquet: 22,865,618 rows, 2010-01-04 to 2022-05-16
crsp_daily_ext.parquet: 9,695,232 rows, 2022-01-03 to 2025-12-31


## Step 5 — Extended moneyness

Joins extended CBOE options against extended CRSP prices. Both inputs
redirected via `cboe_dir` and `crsp_path` (the latter added to
`build_moneyness` for exactly this). Writes `cboe_daily_moneyness_ext/`.

**Expect 2026 to be unpriced.** CRSP ends 2025-12-31, so options traded
Jan–May 2026 have no spot price and classify as UNKNOWN. 2022–2025 should
price at ~87–88%; 2026 at 0%. That pattern is the confirmation the extended
CRSP join took effect — not a bug.

In [10]:
from pipeline.build_moneyness import build_moneyness

ext_mny = DATA_DIR / "cboe_daily_moneyness_ext"
if not list(ext_mny.glob("*.parquet")):
    build_moneyness(
        cboe_dir=DATA_DIR / "cboe_parquet_ext",
        crsp_path=CRSP_DIR / "crsp_daily_ext.parquet",
        out_dir=ext_mny,
    )
else:
    print("cboe_daily_moneyness_ext already exists -- skipping.")

cboe_daily_moneyness_ext already exists -- skipping.


### Verify: schema parity, and 2023 genuinely classified (not all UNKNOWN)

In [11]:
base = pl.read_parquet(DATA_DIR / "cboe_daily_moneyness" / "daily_moneyness_2021.parquet")
ext  = pl.read_parquet(DATA_DIR / "cboe_daily_moneyness_ext" / "daily_moneyness_2023.parquet")
print("column diff:", set(base.columns) ^ set(ext.columns))
print("base files:", len(list((DATA_DIR/"cboe_daily_moneyness").glob("*.parquet"))),
      "| ext files:", len(list((DATA_DIR/"cboe_daily_moneyness_ext").glob("*.parquet"))))
otm = [c for c in ext.columns if "_otm_" in c]
print("2023 total OTM volume (should be large, non-zero):", ext.select(otm).sum_horizontal().sum())

column diff: set()
base files: 12 | ext files: 5
2023 total OTM volume (should be large, non-zero): 475075742


## Step 6 — Confirm the sample switch reads the extension

`set_sample("ext")` redirects every analysis loader — DiD panels, market cap,
moneyness, CRSP frames, the proxy and order-size modules — to the `*_ext`
files. This is the bridge between the extended data (built above) and the
analysis in B3. It restores to base afterwards.

In [12]:
from analysis.event_window_profile import set_sample, current_sample
from analysis.compare_retail_proxies import compare_proxies
from analysis.analyse_order_size import order_size_by_year

print("default sample:", current_sample())
set_sample("ext")
print("\n=== proxies, extended (years should be 2022-2026) ===")
print(compare_proxies())
print("\n=== order size, extended ===")
print(order_size_by_year())
set_sample("base")
print("\nrestored to:", current_sample())

default sample: base
Sample set to 'ext'. Data loads from *_ext files.

=== proxies, extended (years should be 2022-2026) ===
shape: (5, 6)
┌──────┬──────────┬──────────────────┬────────────────────┬────────────┬─────────────┐
│ year ┆ recall   ┆ contamination_lb ┆ classifiable_share ┆ retail_vol ┆ procust_vol │
│ ---  ┆ ---      ┆ ---              ┆ ---                ┆ ---        ┆ ---         │
│ i32  ┆ f64      ┆ f64              ┆ f64                ┆ i64        ┆ i64         │
╞══════╪══════════╪══════════════════╪════════════════════╪════════════╪═════════════╡
│ 2022 ┆ 0.682209 ┆ 0.04176          ┆ 0.447669           ┆ 644430093  ┆ 52323038    │
│ 2023 ┆ 0.669875 ┆ 0.049366         ┆ 0.440338           ┆ 1635086884 ┆ 127600059   │
│ 2024 ┆ 0.665727 ┆ 0.061166         ┆ 0.441274           ┆ 1747057857 ┆ 165495528   │
│ 2025 ┆ 0.679957 ┆ 0.06235          ┆ 0.444459           ┆ 2198256838 ┆ 182604533   │
│ 2026 ┆ 0.688062 ┆ 0.058735         ┆ 0.445456           ┆ 1101195360 ┆ 8234

## Extended sample built

All five extended tables now exist alongside base, schema-verified and
date-checked:

| Table | Base | Extension |
|---|---|---|
| CBOE parquet | `cboe_parquet/` | `cboe_parquet_ext/` |
| IBES events | `dispersion_events.parquet` | `dispersion_events_ext.parquet` |
| CRSP | `crsp_daily.parquet` | `crsp_daily_ext.parquet` |
| Daily retail | `cboe_daily_retail/` | `cboe_daily_retail_ext/` |
| Moneyness | `cboe_daily_moneyness/` | `cboe_daily_moneyness_ext/` |

The post-COVID era analysis is in **B3**.